# 02. YOLOv8n 학습 (베이스라인)

| 선택 | 값 | 이유 |
| --- | --- | --- |
| 모델 | YOLOv8n | 가장 가볍고 Hailo에서 검증된 모델 |
| 입력 크기 | 960 (추론 시 960x288) | KITTI 비율에 맞춘 균형형 |

- 학습 결과는 **Google Drive**에 저장돼요. 접속이 끊겨도 다시 실행하면 **이어서** 학습해요.
- 런타임 유형이 **T4 GPU**인지 확인하고 시작하세요 (런타임 → 런타임 유형 변경).
- 처음부터 끝까지 ▶ 를 차례로 누르면 돼요.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive 연결

권한 허용 창이 뜨면 허용해 주세요. 학습 결과가 `내 드라이브/embeded_ondeviceAI/runs` 에 저장돼요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS = '/content/drive/MyDrive/embeded_ondeviceAI/runs'
import os
os.makedirs(RUNS, exist_ok=True)

## 3. 데이터 준비

01 노트북과 같은 작업이에요. 이미 이번 접속에서 01을 실행했다면 금방 끝나요.

In [ ]:
%%bash
git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /content/embeded_ondeviceAI 2>/dev/null || git -C /content/embeded_ondeviceAI pull -q
mkdir -p /content/kitti && cd /content/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  [ -f $f ] || wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
  unzip -q -n $f 'training/*'
done
cd /content/embeded_ondeviceAI
[ -f /content/kitti_yolo/kitti.yaml ] || python scripts/prepare_kitti.py --kitti /content/kitti --out /content/kitti_yolo

## 4. Ultralytics 설치

출력되는 버전을 보고서의 '실험 환경'에 적어 두세요.

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

## 5. 학습

- 대략 2~3시간 걸려요. 중간중간 화면을 눌러 주면 방치로 끊기는 걸 막을 수 있어요.
- 끊기면 1~4번을 다시 실행한 뒤 이 칸을 실행하면 **저장된 지점부터 이어서** 학습해요.
- `dev` 셋(쪽지시험)으로만 중간 점검해요. 시험지(eval)는 쓰지 않아요.
- 여기 나오는 mAP50 / mAP50-95 는 COCO 방식이라 **대회 점수(KITTI AP40)와 달라요.** 공부가 잘 되는지 보는 용도예요.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

NAME = 'v8n_960'
last = Path(RUNS) / NAME / 'weights' / 'last.pt'

if last.exists():
    print('이어서 학습:', last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO('yolov8n.pt').train(
        data='/content/kitti_yolo/kitti.yaml',
        imgsz=960,
        epochs=60,
        patience=20,      # 20바퀴 동안 나아지지 않으면 일찍 멈춤
        batch=32,
        cache='ram',      # 메모리가 부족하면 자동으로 꺼져요
        seed=0,
        deterministic=True,
        project=RUNS,
        name=NAME,
        exist_ok=True,
    )

## 6. 결과 확인

학습 곡선과 dev 셋 점수를 봐요. 제일 좋은 모델은 `weights/best.pt` 로 Drive에 저장돼 있어요. 이게 **베이스라인 모델**이에요.

In [ ]:
from IPython.display import Image, display
run = Path(RUNS) / NAME
display(Image(str(run / 'results.png'), width=1000))

metrics = YOLO(str(run / 'weights' / 'best.pt')).val(data='/content/kitti_yolo/kitti.yaml', imgsz=960, split='val', batch=32)
for i, n in metrics.names.items():
    print(f'{n:<11} mAP50 {metrics.box.ap50[i]:.3f}')